# 🔧 題目 1：零售 POS 銷售分析
# Mini Data Pipeline 工作坊

> **情境**：你是一家零售連鎖集團的資料顧問。老闆想知道哪些商品最暢銷、哪些客戶最有價值、各國市場表現如何。
>
> **Pipeline**：`CSV → pandas → SQLite (raw/cleaned/analyzed) → SQL → LLM → FastAPI → Streamlit`
>
> **資料**：[Kaggle: Online Retail II UCI](https://www.kaggle.com/datasets/mashlyn/online-retail-ii-uci)（2,000 筆取樣）
>
> 📄 詳細需求見 `requirements_spec.md`

---

### 📋 今日目標

| 必做（Section 1-7） | 選做（Section 8-10） |
|------|------|
| ✅ ETL pipeline（CSV → SQLite 三表） | ⭐ FastAPI API |
| ✅ SQL 查詢統計分析 | ⭐ ipywidgets / Streamlit Dashboard |
| ✅ LLM 品類分類 | ⭐ 本地部署 |
| ✅ 顧問報告 output/report.md | |

### 🗺️ 標記說明

| 標記 | 意思 | 難度 |
|------|------|------|
| `🟢 簡單` | 開放式 — 只給提示，自己寫 | 基礎 pandas 操作 |
| `🟡 中等` | 半骨架 — 給結構，填關鍵處 | 日期轉換、SQL 查詢 |
| `🔴 較難` | 完整骨架 — 給結構填空 | FastAPI、ipywidgets、LLM 批次 |
| `（不需要改）` | 直接跑 | 環境設定、檢查點 |


## Section 0：環境設定

直接跑下面兩格，不需要改。


In [ ]:
# （不需要改）
import pandas as pd
import sqlite3
import os
import json
print("✅ 套件載入完成")


In [ ]:
# （不需要改）API Key 設定
OPENAI_API_KEY = ""
if os.path.exists(".env"):
    with open(".env") as f:
        for line in f:
            if line.startswith("OPENAI_API_KEY"):
                OPENAI_API_KEY = line.strip().split("=", 1)[1]
print("✅ API Key 已設定" if OPENAI_API_KEY else "⚠️ 無 API Key，使用 fallback（不影響完成度）")


---
## Section 1：Extract — 讀取資料 + 寫入 raw 表

> pipeline 第一步：**資料進入系統**。讀 CSV → 檢查 → 寫入 SQLite。


### Step 1-1 🟢 簡單：讀取 CSV

> 💡 方法：`pd.read_csv("檔案路徑")`
> ✅ 預期：2,000 筆、9 個欄位


In [ ]:
# TODO 🟢: 讀取 data/raw/topic_1/orders.csv，存成 df_raw，印出筆數和欄位，看前 5 筆


### Step 1-2 🟢 簡單：檢查資料品質

> 💡 方法：`.dtypes`、`.isnull().sum()`、`.describe()`
> ✅ 預期：看到型別、缺漏值數量、數值範圍


In [ ]:
# TODO 🟢: 用三個方法檢查 df_raw


### Step 1-3 🟢 簡單：自由探索

> 💡 靈感：`value_counts()` 看分佈、`.nunique()` 看有幾種、`.sample(5)` 隨機看 5 筆


In [ ]:
# TODO 🟢: 自由探索，寫你想看的東西


### Step 1-4 🟡 中等：建立 SQLite + 寫入 raw 表

> 💡 方法：`sqlite3.connect()` / `df.to_sql()` / `pd.read_sql()`
> ✅ 預期：印出「raw_orders: 2000 筆」


In [ ]:
# TODO 🟡: 建立 pipeline.db，把 df_raw 寫入 raw_orders 表，驗證筆數
DB_PATH = "pipeline.db"
conn = sqlite3.connect(DB_PATH)

# 寫入 raw 表


# 用 SQL 驗證筆數


---
## Section 2：Transform — 清洗 + 寫入 cleaned 表

> 從**資料庫**讀出 → 清洗 → 寫回資料庫。


### Step 2-1 🟢 簡單：從 raw 表讀出

> 💡 方法：`pd.read_sql("SELECT * FROM 表名", conn)`
> ✅ 預期：df 有 2000 筆


In [ ]:
# TODO 🟢: 從 raw_orders 讀出資料，存成 df，記下筆數


### Step 2-2 🟢 簡單：處理缺漏值

> 💡 方法：`df.dropna(subset=["欄位1", "欄位2"])`
> 🎯 刪除 description 和 customer_id 為空的列
> ✅ 預期：筆數可能減少幾筆


In [ ]:
# TODO 🟢: 刪除缺漏值，印出前後筆數


### Step 2-3 🟡 中等：日期轉換 + 新增時間特徵

> 💡 方法：`pd.to_datetime()` 轉日期、`.dt.year` / `.dt.month` / `.dt.day_name()` / `.dt.hour` 提取
> 🎯 把 invoice_date 轉日期，新增 year, month, day_of_week, hour 四欄
> ✅ 預期：df 多了 4 個欄位


In [ ]:
# TODO 🟡: 日期轉換 + 提取 4 個時間特徵
df["invoice_date"] = pd.to_datetime(df["invoice_date"])
df["year"] = 
df["month"] = 
df["day_of_week"] = 
df["hour"] = 

print(df[["invoice_date", "year", "month", "day_of_week", "hour"]].head(3))


### Step 2-4 🟢 簡單：計算總金額 + 過濾異常值

> 💡 方法：`df["新欄位"] = df["A"] * df["B"]` 計算、`df = df[條件]` 過濾
> 🎯 確認 total_amount = quantity × unit_price，過濾 quantity ≤ 0 或 unit_price ≤ 0
> ✅ 預期：total_amount 全是正數


In [ ]:
# TODO 🟢: 計算總金額 + 過濾異常值


### 🏁 清洗檢查點

> 直接跑。全部 ✅ 才往下。


In [ ]:
# （不需要改）
assert df.isnull().sum().sum() == 0, "❌ 還有缺漏值！回去看 Step 2-2"
assert (df["quantity"] > 0).all(), "❌ quantity 有非正值！回去看 Step 2-4"
assert (df["unit_price"] > 0).all(), "❌ unit_price 有非正值！回去看 Step 2-4"
assert "year" in df.columns, "❌ 缺少 year！回去看 Step 2-3"
assert "month" in df.columns, "❌ 缺少 month！回去看 Step 2-3"
print("✅ 全部檢查通過！")
print(f"   清洗後: {len(df)} 筆, {len(df.columns)} 欄")


### Step 2-5 🟢 簡單：寫入 cleaned 表

> 💡 方法：跟 Step 1-4 一樣，表名改 `cleaned_orders`
> ✅ 預期：cleaned_orders 筆數 ≤ raw_orders


In [ ]:
# TODO 🟢: 寫入 cleaned_orders 表，驗證兩表筆數


---
## Section 3：SQL 統計分析

> 用 SQL 從資料庫做分析。
> 要回答：1. 商品銷售排行？ 2. 各國營收？ 3. 你自己想知道什麼？


### Step 3-1 🟡 中等：商品銷售排行

> 💡 SQL 骨架：`SELECT 欄位, 聚合函式 FROM 表 GROUP BY 欄位 ORDER BY ... DESC LIMIT N`
> ✅ 預期：Top 20 商品和銷售額


In [ ]:
# TODO 🟡: 商品銷售排行（description, 訂單數, 總數量, 總金額 Top 20）
product_stats = pd.read_sql("""
    SELECT description,
           COUNT(*) as order_count,
           SUM(quantity) as total_qty,
           ROUND(SUM(                ), 2) as total_revenue
    FROM 
    GROUP BY 
    ORDER BY              DESC
    LIMIT 
""", conn)
product_stats


### Step 3-2 🟡 中等：各國銷售統計

> 💡 提示：`COUNT(DISTINCT customer_id)` 算不重複客戶數
> ✅ 預期：United Kingdom 營收最高


In [ ]:
# TODO 🟡: 各國客戶數、訂單數、營收
country_stats = pd.read_sql("""
    SELECT country,
           
           
           
    FROM cleaned_orders
    GROUP BY 
    ORDER BY              DESC
""", conn)
country_stats


### Step 3-3 🟡 中等：視覺化

> 💡 方法：`df.plot.barh(x="欄位", y="值")` 或 `df.plot.bar()`
> 🎯 把上面的統計結果畫成至少一張圖
> ✅ 預期：看到清楚的圖表


In [ ]:
# TODO 🟡: 視覺化統計結果
import matplotlib.pyplot as plt


### Step 3-4 🟢 簡單：自由探索 SQL

> 💡 靈感：`GROUP BY hour`（時段）、`GROUP BY day_of_week`（星期）、`GROUP BY customer_id`（客戶排行）


In [ ]:
# TODO 🟢: 你自己的 SQL 查詢


### Step 3-5 🟢 簡單：存統計結果

> 💡 方法：`os.makedirs()` 建資料夾、`df.to_csv()` 存檔
> ✅ 預期：data/processed/ 裡有 CSV


In [ ]:
# TODO 🟢: 存統計結果到 data/processed/


---
## Section 4：LLM 加值分析

> 用 LLM 對商品描述做品類分類。
> helper 函式已寫好，你要做的是：呼叫、看結果、跑批次、寫入資料庫。


In [ ]:
# （不需要改）LLM Helper 函式
import requests

def llm_analyze(text, api_key=None):
    if api_key:
        return _llm_api(text, api_key)
    return _llm_fallback(text)

def _llm_api(text, api_key):
    prompt = f"""請分析以下零售商品描述，回傳 JSON：
{{"category": "家飾/禮品/餐具/季節商品/文具/其他", "insight": "一句話商品洞察"}}
商品描述：{text[:300]}"""
    try:
        resp = requests.post("https://api.openai.com/v1/chat/completions",
            headers={"Authorization": f"Bearer {api_key}"},
            json={"model": "gpt-4o-mini", "messages": [{"role": "user", "content": prompt}], "temperature": 0.3},
            timeout=30)
        content = resp.json()["choices"][0]["message"]["content"].strip()
        if content.startswith("```"): content = content.split("\n", 1)[1].rsplit("```", 1)[0]
        return json.loads(content)
    except:
        return _llm_fallback(text)

def _llm_fallback(text):
    t = text.lower()
    if any(w in t for w in ["christmas", "xmas", "santa", "winter"]): cat = "季節商品"
    elif any(w in t for w in ["candle", "holder", "frame", "lamp"]): cat = "家飾"
    elif any(w in t for w in ["cup", "mug", "plate", "bowl"]): cat = "餐具"
    elif any(w in t for w in ["pen", "pencil", "notebook", "card"]): cat = "文具"
    elif any(w in t for w in ["gift", "bag", "box", "ribbon"]): cat = "禮品"
    else: cat = "其他"
    return {"category": cat, "insight": text[:50] + "..."}

print("✅ LLM Helper 已定義")


### Step 4-1 🟢 簡單：單筆測試

> 💡 方法：`df["description"].iloc[0]` 取一筆、`llm_analyze(文字, api_key)` 呼叫
> ✅ 預期：回傳 dict 有 category 和 insight


In [ ]:
# TODO 🟢: 取一筆 description，呼叫 llm_analyze，印出結果


### Step 4-2 🟡 中等：批次分析

> 💡 方法：for 迴圈 + `df.head(N).iterrows()` + `results.append()`
> 🎯 先跑 50 筆
> ⏱ fallback 幾秒；API 版 50 筆約 2-3 分鐘


In [ ]:
# TODO 🟡: 批次分析前 50 筆
BATCH_SIZE = 50
api_key = OPENAI_API_KEY if OPENAI_API_KEY else None

results = []
for i, row in df.head(BATCH_SIZE).iterrows():
    r = llm_analyze(              , api_key)
    results.append(r)
    if len(results) % 10 == 0:
        print(f"  進度: {len(results)}/{BATCH_SIZE}")

print(f"✅ 完成 {len(results)} 筆")


### Step 4-3 🟡 中等：整理結果 + 寫入 analyzed 表

> 💡 方法：
> - `df.head(N).copy()` 複製
> - `[r["key"] for r in results]` 從 list of dict 提取
> - `to_sql()` 寫入
> ✅ 預期：三表都有資料


In [ ]:
# TODO 🟡: 整理結果，加 category 和 llm_insight 欄位，寫入 analyzed_orders 表
df_analyzed = df.head(BATCH_SIZE).copy()
df_analyzed["category"] = 
df_analyzed["llm_insight"] = 

# 寫入 analyzed 表


# 印出三表筆數確認


---
## Section 5：驗證 pipeline


### Step 5-1 🟡 中等：跨表查詢

> 💡 SQL 語法：`SELECT '名稱' as layer, COUNT(*) as rows FROM 表 UNION ALL SELECT ...`
> ✅ 預期：raw ≥ cleaned ≥ analyzed


In [ ]:
# TODO 🟡: 用 UNION ALL 一次查三表筆數
lineage = pd.read_sql("""
    SELECT 'raw_orders' as layer, COUNT(*) as rows FROM raw_orders
    UNION ALL
    SELECT                   ,               FROM 
    UNION ALL
    SELECT                   ,               FROM 
""", conn)
print("📊 Pipeline 資料流：")
print(lineage.to_string(index=False))


---
## Section 6：產出報告


### Step 6-1 🟢 簡單：寫報告

> 💡 方法：f-string 嵌入變數，存到 output/report.md
> 🎯 報告要有具體數字和建議，不是空話
> ✅ 預期：output/report.md 存在且有內容


In [ ]:
# TODO 🟢: 寫報告 — 查數字、填內容、存檔
report = f"""# 零售 POS 銷售分析報告

## 資料概要
（填入：分析筆數、總銷售額、資料來源）

## 關鍵發現
（根據 Section 3 統計結果，寫 2-3 個發現）

## 品類分佈
（根據 Section 4 LLM 分析，列出分佈）

## 建議
（寫 2-3 條有數據支撐的建議）

## Pipeline
CSV → pandas → SQLite(raw/cleaned/analyzed) → SQL → LLM → 本報告
"""

os.makedirs("output", exist_ok=True)
with open("output/report.md", "w") as f:
    f.write(report)
print("✅ 報告已存到 output/report.md")


---
## Section 7：打包確認

> 直接跑。


In [ ]:
# （不需要改）
checks = [("pipeline.db", "SQLite 資料庫"), ("data/processed", "統計結果"), ("output/report.md", "顧問報告")]
print("📋 產出確認：")
all_ok = True
for path, desc in checks:
    exists = os.path.exists(path)
    print(f"  {'✅' if exists else '❌'} {desc}: {path}")
    if not exists: all_ok = False
if os.path.exists("pipeline.db"):
    c = sqlite3.connect("pipeline.db")
    for t in ["raw_orders", "cleaned_orders", "analyzed_orders"]:
        try:
            n = pd.read_sql(f"SELECT COUNT(*) as n FROM {t}", c)["n"][0]
            print(f"  ✅ {t}: {n} 筆")
        except:
            print(f"  ❌ {t} 不存在"); all_ok = False
    c.close()
print("\n🎉 全部完成！" if all_ok else "\n⚠️ 有缺漏，請回去補完。")
print("\n📋 接下來：README + upgrade_plan + 3 分鐘 Demo + Section 8-10")


---
## Section 8（選做）：FastAPI — 把分析結果變成 API

> **為什麼？** 別人不能打開你的 .db 檔。API 把結果包裝成網址。
>
> **兩條路線**：🅰️ 在下面寫 / 🅱️ 開 `api.py`（那就是 solution）


### Step 8-1 🔴 較難：定義 endpoint

> 💡 FastAPI 骨架已給，填入 SQL 查詢和回傳格式
> 🎯 定義 3 個 endpoint：/health、/stats、/analyzed


In [ ]:
# TODO 🔴: 定義 FastAPI endpoint（骨架已給，填空處）

# 安裝（直接跑）
!pip install -q fastapi uvicorn nest_asyncio
from fastapi import FastAPI
import nest_asyncio
nest_asyncio.apply()

# 建立 app
api = FastAPI(title="零售銷售分析 API")

# /health — 回傳狀態
@api.get("/health")
def health():
    return {"status": "ok"}

# TODO: /stats — 從 cleaned_orders 查商品銷售排行
@api.get("/stats")
def get_stats():
    c = sqlite3.connect("pipeline.db")
    df = pd.read_sql("""
        SELECT description,
               COUNT(*) as order_count,
               ROUND(SUM(              ), 2) as total_revenue
        FROM 
        GROUP BY 
        ORDER BY total_revenue DESC
        LIMIT 20
    """, c)
    c.close()
    return df.to_dict(orient="records")

# TODO: /analyzed — 從 analyzed_orders 查 LLM 結果
@api.get("/analyzed")
def get_analyzed():
    c = sqlite3.connect("pipeline.db")
    df = pd.read_sql("""
        SELECT              ,              ,
        FROM 
        LIMIT 20
    """, c)
    c.close()
    return df.to_dict(orient="records")

print("✅ API 定義完成")


### Step 8-2 🟡 中等：啟動 + 測試

> 💡 方法：`requests.get("http://localhost:8000/路徑").json()`
> ✅ 預期：每個 endpoint 回傳 JSON


In [ ]:
# 背景啟動（直接跑）
import threading, uvicorn, time
thread = threading.Thread(target=uvicorn.run, args=(api,), kwargs={"host": "0.0.0.0", "port": 8000, "log_level": "warning"})
thread.daemon = True
thread.start()
time.sleep(2)
print("✅ API 已啟動")

# TODO 🟡: 用 requests 測試 3 個 endpoint
import requests


### 路線 🅱️

`api.py` 是這個 Section 的 **solution**。本地跑：`uvicorn api:app --reload --port 8000`


---
## Section 9（選做）：Dashboard — 視覺化

> 🅰️ 在下面用 ipywidgets / 🅱️ 開 `app.py`（Streamlit，solution）


### Step 9-1 🔴 較難：互動 Dashboard

> 💡 骨架已給，填入：資料來源、選單選項、篩選邏輯、畫圖欄位
> 🎯 目標：選國家 → 自動更新商品銷售排行圖


In [ ]:
# TODO 🔴: 互動 Dashboard（骨架已給，填空處）
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

# 讀資料
df_dash = pd.read_sql("SELECT * FROM              ", conn)

# 下拉選單
country_dropdown = widgets.Dropdown(
    options=["全部"] + sorted(df_dash["         "].unique().tolist()),
    description="選國家："
)

# 更新函式
def update_dashboard(country):
    clear_output(wait=True)
    display(country_dropdown)
    
    # 篩選
    if country == "全部":
        data = df_dash
    else:
        data = df_dash[df_dash["         "] == country]
    
    # 統計
    print(f"📊 {country}: {len(data)} 筆, 營收 ${data['             '].sum():,.2f}")
    
    # 畫圖
    fig, ax = plt.subplots(figsize=(10, 5))
    data.groupby("              ")["             "].sum().sort_values().tail(10).plot.barh(ax=ax)
    ax.set_title(f"商品銷售額 Top 10 — {country}")
    plt.tight_layout()
    plt.show()

# 綁定
widgets.interact(update_dashboard, country=country_dropdown)


### 🎯 進階：加更多互動

| 想加什麼 | 做法 |
|---------|------|
| 第二個選單 | 再建一個 `Dropdown` |
| Top N 滑桿 | `widgets.IntSlider(min=1, max=50)` |
| 多圖並排 | `plt.subplots(1, 2)` |


### 路線 🅱️

`app.py` 是 solution。本地跑：`streamlit run app.py`


---
## Section 10（選做）：本地部署指引

```bash
# API
cd data/raw/topic_1
uvicorn api:app --reload --port 8000

# Dashboard（另開 terminal）
streamlit run app.py
```

### 後續升級

| 現在 | 升級後 | 對應課程 |
|------|--------|---------|
| SQLite | MySQL / BigQuery | 資料庫模組 |
| 手動跑 | Airflow DAG | Airflow 模組 |
| 本地 Streamlit | Docker 容器化 | Docker 模組 |
| 本地開發 | GCP 雲端部署 | GCP 模組 |
